In [1]:
# !pip install growwapi
# !pip install --upgrade growwapi
# !pip install pyarrow fastparquet

In [2]:
import pandas as pd
import numpy as np
import math
import os
from datetime import datetime, timedelta
from growwapi import GrowwAPI

In [6]:
API_Key = "eyJraWQiOiJaTUtjVXciLCJhbGciOiJFUzI1NiJ9.eyJleHAiOjI1NjE2OTk1NTcsImlhdCI6MTc3MzI5OTU1NywibmJmIjoxNzczMjk5NTU3LCJzdWIiOiJ7XCJ0b2tlblJlZklkXCI6XCI0NjI5ODEwMi1lNzViLTQ5NTEtOTg5NC02MDEyYzk4ZTY1ZWJcIixcInZlbmRvckludGVncmF0aW9uS2V5XCI6XCJlMzFmZjIzYjA4NmI0MDZjODg3NGIyZjZkODQ5NTMxM1wiLFwidXNlckFjY291bnRJZFwiOlwiODdlODE1ZjAtZWM5MS00OTU4LWIwNzUtMDg0MWMyMTk2MzNkXCIsXCJkZXZpY2VJZFwiOlwiNTEyN2ZiZjktMTZiMS01NzUzLTk2MjgtOTQyMTQwNGJhZDYwXCIsXCJzZXNzaW9uSWRcIjpcIjk1ZWE2OWNiLWVlYTgtNDBlYy04ZDA4LTZmMWQzNTdhMjNiMVwiLFwiYWRkaXRpb25hbERhdGFcIjpcIno1NC9NZzltdjE2WXdmb0gvS0EwYkFvc2NJMXRRNzBlTXdSeHhOa1IxMTlSTkczdTlLa2pWZDNoWjU1ZStNZERhWXBOVi9UOUxIRmtQejFFQisybTdRPT1cIixcInJvbGVcIjpcImF1dGgtdG90cFwiLFwic291cmNlSXBBZGRyZXNzXCI6XCI0OS4yNDguMTAyLjE1OCwxNzIuNjkuODcuNTYsMzUuMjQxLjIzLjEyM1wiLFwidHdvRmFFeHBpcnlUc1wiOjI1NjE2OTk1NTczMTl9IiwiaXNzIjoiYXBleC1hdXRoLXByb2QtYXBwIn0.0RHgeDxBY7MNie9aX1taFj-DSg_BmsMrsO24qFSqaaIbvHp8i2sP9delLcIs39KnRdI9YLDgWEqZzir_pxEdnA"
Secret_Key = "@AV21R#iv30DuYLHd4d$Q9YsuSDvNCcT"



In [7]:
access_token = GrowwAPI.get_access_token(api_key=API_Key, secret=Secret_Key)
# Use access_token to initiate GrowwAPI
groww = GrowwAPI(access_token)

Ready to Groww!


In [8]:
symbol_segment_map = [
    ("RELIANCE", "Oil Gas & Consumable Fuels"), ("HDFCBANK", "Financial Services"), ("ICICIBANK", "Financial Services"),
    ("INFY", "Information Technology"), ("TCS", "Information Technology"), ("HINDUNILVR", "FMCG"), ("ITC", "FMCG"),
    ("BHARTIARTL", "Telecommunication"), ("SBIN", "Financial Services"), ("KOTAKBANK", "Financial Services"),
    ("AXISBANK", "Financial Services"), ("BAJFINANCE", "Financial Services"), ("BAJAJFINSV", "Financial Services"),
    ("HCLTECH", "Information Technology"), ("WIPRO", "Information Technology"), ("LTIM", "Information Technology"),
    ("TECHM", "Information Technology"), ("LT", "Construction"), ("ULTRACEMCO", "Construction Materials"),
    ("GRASIM", "Construction Materials"), ("SHREECEM", "Construction Materials"), ("TATASTEEL", "Metals & Mining"),
    ("JSWSTEEL", "Metals & Mining"), ("HINDALCO", "Metals & Mining"), ("COALINDIA", "Oil Gas & Consumable Fuels"),
    ("ONGC", "Oil Gas & Consumable Fuels"), ("POWERGRID", "Power"), ("NTPC", "Power"), ("ADANIENT", "Diversified"),
    ("ADANIPORTS", "Services"), ("ASIANPAINT", "Consumer Durables"), ("TITAN", "Consumer Durables"), ("NESTLEIND", "FMCG"),
    ("BRITANNIA", "FMCG"), ("MARUTI", "Automobile"), ("M&M", "Automobile"), ("TATAMOTORS", "Automobile"),
    ("BAJAJ-AUTO", "Automobile"), ("HEROMOTOCO", "Automobile"), ("EICHERMOT", "Automobile"), ("APOLLOHOSP", "Healthcare"),
    ("SUNPHARMA", "Healthcare"), ("DRREDDY", "Healthcare"), ("CIPLA", "Healthcare"), ("DIVISLAB", "Healthcare"),
    ("UPL", "Chemicals"), ("SBILIFE", "Financial Services"), ("HDFCLIFE", "Financial Services"), ("SBICARD", "Financial Services"),
    ("INDUSINDBK", "Financial Services")]

symbol_segment_df = pd.DataFrame(symbol_segment_map, columns=["Symbol", "Sector"])

print(symbol_segment_df)

        Symbol                      Sector
0     RELIANCE  Oil Gas & Consumable Fuels
1     HDFCBANK          Financial Services
2    ICICIBANK          Financial Services
3         INFY      Information Technology
4          TCS      Information Technology
5   HINDUNILVR                        FMCG
6          ITC                        FMCG
7   BHARTIARTL           Telecommunication
8         SBIN          Financial Services
9    KOTAKBANK          Financial Services
10    AXISBANK          Financial Services
11  BAJFINANCE          Financial Services
12  BAJAJFINSV          Financial Services
13     HCLTECH      Information Technology
14       WIPRO      Information Technology
15        LTIM      Information Technology
16       TECHM      Information Technology
17          LT                Construction
18  ULTRACEMCO      Construction Materials
19      GRASIM      Construction Materials
20    SHREECEM      Construction Materials
21   TATASTEEL             Metals & Mining
22    JSWST

### Historical Data Storing

In [13]:
def get_historical_data(start_date, end_date, symbol="RELIANCE"):
    historical_data_response = groww.get_historical_candles(
        groww_symbol=f"NSE-{symbol.upper()}",
        exchange=groww.EXCHANGE_NSE,
        segment=groww.SEGMENT_CASH,
        start_time=start_date,
        end_time=end_date,
        candle_interval=groww.CANDLE_INTERVAL_DAY # Optional: Interval in days for the candle data
    )
    return historical_data_response

In [16]:
def fetch_and_store_hist_data(symbol):    
    file_path = os.path.join(os.getcwd(), f"{symbol}_Hist_Data.csv")
    
    if os.path.exists(file_path):
        print(f"Data already exists for {symbol}")
        
    else:
        start_date = pd.to_datetime('2020-01-01')
        end_date = pd.to_datetime('2026-01-01')
        total_days = end_date - start_date
        data_list = []
        while total_days > timedelta(days=0):
            intermediate_end_date = start_date + timedelta(days=180) if total_days > timedelta(days=180) else start_date + timedelta(days=total_days.days)
            print(start_date, " => ", intermediate_end_date)
            hist_data = get_historical_data(start_date, intermediate_end_date, symbol)
            print(hist_data["candles"], '\n\n')
            if hist_data["candles"]:
                data_list.append(hist_data["candles"])
            start_date = intermediate_end_date
            total_days = total_days - timedelta(days=180)
        #     print(total_days, '\n')

        data_list = np.concatenate(data_list)
        data_list

        df = pd.DataFrame(
            data_list,
            columns=["Date", "Open", "High", "Low", "Close", "Volume", "Extra"]
        )

        df.to_csv(f"{symbol}_Hist_Data.csv")

In [17]:
stock_df = fetch_and_store_hist_data("LTIM")
stock_df

Data already exists for LTIM


In [18]:
for symbol in symbol_segment_df['Symbol']:
    print(f"Fetching data for {symbol} \n")
    stock_df = fetch_and_store_hist_data(symbol)
    

Fetching data for RELIANCE 

Data already exists for RELIANCE
Fetching data for HDFCBANK 

Data already exists for HDFCBANK
Fetching data for ICICIBANK 

Data already exists for ICICIBANK
Fetching data for INFY 

Data already exists for INFY
Fetching data for TCS 

Data already exists for TCS
Fetching data for HINDUNILVR 

Data already exists for HINDUNILVR
Fetching data for ITC 

Data already exists for ITC
Fetching data for BHARTIARTL 

Data already exists for BHARTIARTL
Fetching data for SBIN 

Data already exists for SBIN
Fetching data for KOTAKBANK 

Data already exists for KOTAKBANK
Fetching data for AXISBANK 

Data already exists for AXISBANK
Fetching data for BAJFINANCE 

Data already exists for BAJFINANCE
Fetching data for BAJAJFINSV 

Data already exists for BAJAJFINSV
Fetching data for HCLTECH 

Data already exists for HCLTECH
Fetching data for WIPRO 

Data already exists for WIPRO
Fetching data for LTIM 

Data already exists for LTIM
Fetching data for TECHM 

2020-01-01 00:

[['2020-01-01T00:00:00', 766.0, 766.8, 760.35, 762.1, 745793, None], ['2020-01-02T00:00:00', 762.1, 769.15, 761.0, 766.05, 1133299, None], ['2020-01-03T00:00:00', 766.0, 779.9, 763.55, 775.1, 2121123, None], ['2020-01-06T00:00:00', 774.85, 778.8, 768.15, 770.4, 1734437, None], ['2020-01-07T00:00:00', 770.0, 778.5, 760.6, 777.1, 1936317, None], ['2020-01-08T00:00:00', 770.1, 784.9, 767.1, 769.8, 2596557, None], ['2020-01-09T00:00:00', 770.5, 776.45, 762.2, 773.65, 1678737, None], ['2020-01-10T00:00:00', 779.0, 781.9, 772.45, 776.25, 1105622, None], ['2020-01-13T00:00:00', 779.9, 788.7, 775.05, 785.85, 1391232, None], ['2020-01-14T00:00:00', 785.85, 797.8, 783.7, 796.25, 1628281, None], ['2020-01-15T00:00:00', 796.15, 803.65, 788.5, 795.15, 1759644, None], ['2020-01-16T00:00:00', 793.0, 795.7, 781.1, 782.85, 1109364, None], ['2020-01-17T00:00:00', 783.5, 786.75, 775.5, 778.45, 987978, None], ['2020-01-20T00:00:00', 778.5, 785.45, 774.55, 778.2, 1619508, None], ['2020-01-21T00:00:00', 777

[['2020-06-29T00:00:00', 558.9, 558.9, 539.2, 541.55, 3304433, None], ['2020-06-30T00:00:00', 547.0, 548.9, 541.55, 543.4, 2353661, None], ['2020-07-01T00:00:00', 544.0, 547.35, 536.05, 545.9, 3659290, None], ['2020-07-02T00:00:00', 549.9, 562.5, 542.5, 560.85, 5966470, None], ['2020-07-03T00:00:00', 565.9, 572.25, 561.05, 567.2, 3996761, None], ['2020-07-06T00:00:00', 574.95, 588.0, 573.05, 581.55, 4893457, None], ['2020-07-07T00:00:00', 583.85, 592.5, 577.1, 586.85, 5120292, None], ['2020-07-08T00:00:00', 589.95, 593.9, 576.1, 583.45, 3386729, None], ['2020-07-09T00:00:00', 583.45, 587.7, 572.05, 577.3, 2497748, None], ['2020-07-10T00:00:00', 572.0, 572.0, 561.35, 569.0, 3957715, None], ['2020-07-13T00:00:00', 571.95, 601.8, 570.5, 600.05, 11005706, None], ['2020-07-14T00:00:00', 594.95, 603.8, 592.55, 598.3, 5524583, None], ['2020-07-15T00:00:00', 604.0, 630.0, 602.0, 615.3, 11899283, None], ['2020-07-16T00:00:00', 627.0, 638.95, 588.1, 599.4, 12998997, None], ['2020-07-17T00:00:00'




2020-12-26 00:00:00  =>  2021-06-24 00:00:00


[['2020-12-28T00:00:00', 947.25, 956.45, 944.25, 947.1, 1870662, None], ['2020-12-29T00:00:00', 948.0, 968.25, 943.85, 965.65, 4388711, None], ['2020-12-30T00:00:00', 970.0, 985.0, 967.8, 983.25, 6388965, None], ['2020-12-31T00:00:00', 981.95, 986.4, 968.7, 973.2, 2858547, None], ['2021-01-01T00:00:00', 976.0, 981.2, 971.2, 977.95, 1267617, None], ['2021-01-04T00:00:00', 980.6, 1005.0, 975.0, 1001.95, 3515310, None], ['2021-01-05T00:00:00', 1002.5, 1007.95, 991.3, 1003.85, 4040324, None], ['2021-01-06T00:00:00', 1004.0, 1026.0, 983.35, 997.15, 4677362, None], ['2021-01-07T00:00:00', 1005.5, 1010.0, 988.6, 994.75, 3354895, None], ['2021-01-08T00:00:00', 1007.75, 1060.0, 1005.45, 1051.1, 9277307, None], ['2021-01-11T00:00:00', 1062.1, 1081.55, 1040.5, 1077.6, 6487441, None], ['2021-01-12T00:00:00', 1077.6, 1077.65, 1053.0, 1064.2, 4257929, None], ['2021-01-13T00:00:00', 1074.0, 1078.75, 1050.05, 1069.65, 4670656, None], ['2021-01-14T00:00:00', 1064.0, 1064.0, 1030.0, 1052.6, 4765346, Non

...

[['2024-06-10T00:00:00', 1498.5, 1505.8, 1480.1, 1485.85, 2414789, None], ['2024-06-11T00:00:00', 1485.0, 1491.0, 1475.2, 1481.05, 1737488, None], ['2024-06-12T00:00:00', 1491.0, 1492.25, 1473.0, 1484.25, 5419534, None], ['2024-06-13T00:00:00', 1496.0, 1510.0, 1481.2, 1507.25, 2455062, None], ['2024-06-14T00:00:00', 1510.5, 1515.9, 1495.95, 1502.35, 2706721, None], ['2024-06-18T00:00:00', 1514.0, 1516.55, 1499.1, 1507.9, 2811881, None], ['2024-06-19T00:00:00', 1516.2, 1550.0, 1513.25, 1528.2, 11002585, None], ['2024-06-20T00:00:00', 1536.0, 1540.75, 1513.0, 1527.85, 3589444, None], ['2024-06-21T00:00:00', 1523.0, 1537.8, 1511.25, 1527.15, 5257126, None], ['2024-06-24T00:00:00', 1507.15, 1508.9, 1478.2, 1490.4, 4608596, None], ['2024-06-25T00:00:00', 1494.75, 1502.15, 1482.2, 1495.55, 3309487, None], ['2024-06-26T00:00:00', 1498.0, 1521.0, 1481.55, 1497.9, 5604235, None], ['2024-06-27T00:00:00', 1494.0, 1504.95, 1485.0, 1502.75, 5165219, None], ['2024-06-28T00:00:00', 1491.0, 1491.0, 14




2024-12-05 00:00:00  =>  2025-06-03 00:00:00


[['2024-12-05T00:00:00', 1006.0, 1006.0, 990.55, 998.2, 5253901, None], ['2024-12-06T00:00:00', 1000.0, 1003.05, 986.3, 990.35, 6256370, None], ['2024-12-09T00:00:00', 992.8, 994.0, 977.8, 982.6, 3564183, None], ['2024-12-10T00:00:00', 983.0, 988.4, 979.1, 984.3, 3843324, None], ['2024-12-11T00:00:00', 985.5, 999.9, 981.25, 984.85, 4679637, None], ['2024-12-12T00:00:00', 986.0, 1005.5, 986.0, 997.95, 5771168, None], ['2024-12-13T00:00:00', 991.0, 997.65, 965.55, 986.65, 5673427, None], ['2024-12-16T00:00:00', 987.9, 1007.95, 985.55, 999.35, 3056738, None], ['2024-12-17T00:00:00', 1002.0, 1002.0, 972.0, 975.65, 3293065, None], ['2024-12-18T00:00:00', 976.8, 979.3, 960.5, 965.2, 2957750, None], ['2024-12-19T00:00:00', 959.0, 966.85, 948.0, 964.4, 3665412, None], ['2024-12-20T00:00:00', 960.0, 966.85, 926.45, 929.45, 4466134, None], ['2024-12-23T00:00:00', 938.95, 948.8, 930.05, 945.7, 2501963, None], ['2024-12-24T00:00:00', 941.0, 946.0, 929.8, 935.3, 5602517, None], ['2024-12-26T00:00:0

[['2025-06-03T00:00:00', 812.7, 818.45, 798.65, 800.85, 3241313, None], ['2025-06-04T00:00:00', 800.85, 815.45, 800.85, 814.35, 3877545, None], ['2025-06-05T00:00:00', 814.35, 818.0, 800.45, 803.2, 3366237, None], ['2025-06-06T00:00:00', 803.2, 845.85, 803.2, 822.85, 10916786, None], ['2025-06-09T00:00:00', 822.85, 840.0, 821.65, 836.65, 6894267, None], ['2025-06-10T00:00:00', 836.65, 857.0, 836.25, 845.05, 10162134, None], ['2025-06-11T00:00:00', 845.05, 851.4, 834.5, 836.3, 2359273, None], ['2025-06-12T00:00:00', 836.3, 843.15, 826.0, 829.9, 2744911, None], ['2025-06-13T00:00:00', 829.9, 829.9, 813.15, 816.85, 3164561, None], ['2025-06-16T00:00:00', 816.85, 824.5, 810.25, 821.25, 2286257, None], ['2025-06-17T00:00:00', 821.25, 821.25, 806.15, 809.15, 2343961, None], ['2025-06-18T00:00:00', 809.15, 855.5, 809.15, 850.5, 20826809, None], ['2025-06-19T00:00:00', 850.5, 850.5, 835.5, 837.5, 4363823, None], ['2025-06-20T00:00:00', 837.5, 847.5, 829.0, 840.25, 20961217, None], ['2025-06-23




2025-11-30 00:00:00  =>  2026-01-01 00:00:00
[['2025-12-01T00:00:00', None, 861.6, 842.65, 847.15, 3019328, None], ['2025-12-02T00:00:00', None, 860.25, 847.05, 850.3, 2676410, None], ['2025-12-03T00:00:00', None, 856.3, 842.05, 846.9, 2067503, None], ['2025-12-04T00:00:00', None, 872.9, 842.9, 863.0, 4259999, None], ['2025-12-05T00:00:00', None, 874.35, 856.1, 870.1, 3377823, None], ['2025-12-08T00:00:00', None, 873.45, 834.5, 841.4, 2015558, None], ['2025-12-09T00:00:00', None, 848.7, 832.5, 844.35, 3017321, None], ['2025-12-10T00:00:00', None, 851.25, 828.15, 833.85, 2623610, None], ['2025-12-11T00:00:00', None, 842.7, 827.65, 835.55, 2810755, None], ['2025-12-12T00:00:00', None, 851.2, 839.3, 846.15, 3492109, None], ['2025-12-15T00:00:00', None, 853.5, 835.45, 851.25, 2458486, None], ['2025-12-16T00:00:00', None, 856.8, 841.85, 845.05, 2870475, None], ['2025-12-17T00:00:00', None, 846.1, 827.8, 833.85, 3118876, None], ['2025-12-18T00:00:00', None, 842.5, 826.0, 834.9, 1548832, N

In [23]:
df_list = []
for symbol in symbol_segment_df['Symbol']:
    df = pd.read_csv(f"{symbol}_Hist_Data.csv")
    df["Date"] = pd.to_datetime(df["Date"])
    df.set_index("Date", inplace=True)
    
    # remove duplicate dates
    df = df[~df.index.duplicated(keep="first")]
    
    new_df = df["Close"].rename(symbol)
    df_list.append(new_df)


In [27]:
merged_df = pd.concat(df_list, axis=1)
merged_df

,RELIANCE,HDFCBANK,ICICIBANK,INFY,TCS,HINDUNILVR,ITC,BHARTIARTL,SBIN,KOTAKBANK,...,APOLLOHOSP,SUNPHARMA,DRREDDY,CIPLA,DIVISLAB,UPL,SBILIFE,HDFCLIFE,SBICARD,INDUSINDBK
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-01,1509.60,1278.60,536.75,736.85,2167.60,1936.55,238.10,453.30,334.45,1674.05,...,1426.35,434.30,2879.40,475.90,1818.85,588.25,976.40,621.45,NaN,1484.30
2020-01-02,1535.30,1286.75,540.60,734.70,2157.65,1938.05,239.85,455.20,339.30,1671.55,...,1494.65,434.95,2864.90,473.50,1826.85,595.45,969.35,634.25,NaN,1529.00
2020-01-03,1537.15,1268.40,538.85,746.00,2200.65,1927.45,238.50,455.10,333.70,1657.10,...,1486.10,444.60,2883.90,469.95,1834.80,590.75,975.10,630.70,NaN,1528.85
2020-01-06,1501.50,1240.95,525.70,738.85,2200.45,1915.45,235.10,449.65,319.00,1652.55,...,1462.50,439.95,2878.85,466.75,1838.35,584.95,983.10,619.55,NaN,1469.40
2020-01-07,1524.60,1260.60,522.90,727.90,2205.85,1920.70,235.35,445.10,318.40,1670.85,...,1478.70,446.40,2884.20,468.60,1846.85,595.05,984.90,629.95,NaN,1461.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-26,1559.20,992.10,1350.40,1656.10,3280.00,2285.40,404.15,2105.40,966.30,2164.20,...,7156.00,1719.50,1269.30,1506.00,6427.50,774.05,2019.10,748.45,864.60,849.85
2025-12-29,1545.60,991.70,1343.30,1644.70,3251.50,2293.30,402.60,2081.60,965.05,2158.60,...,7084.50,1717.20,1268.60,1494.00,6390.50,770.95,2009.80,746.45,849.95,839.60
2025-12-30,1539.80,990.90,1342.50,1621.60,3246.80,2290.20,400.60,2099.80,973.45,2152.70,...,6990.00,1720.20,1265.80,1492.50,6360.50,787.35,1995.50,743.00,843.40,841.50


In [28]:
merged_df.to_csv("Merged_Data.csv")